# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:\n", metadata.description)
print("Published Date:", metadata.datePublished)
print("License:", metadata.license)
print("Version:", metadata.version)
print("Keywords:", metadata.keywords)
print("Personal Sensitive Info:", metadata.personalSensitiveInformation)


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Examine the record sets and fields in the dataset
record_sets = list(dataset.record_sets())  # retrieves record set objects
print("Record Sets Available:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs['name']}")

# For each record set, list fields and their ids
for rs in record_sets:
    print(f"\nFields for Record Set (@id={rs['@id']}, name={rs['name']}):")
    fields = dataset.fields(record_set=rs['@id'])
    for fld in fields:
        print(f"  - @id: {fld['@id']} | name: {fld.get('name', '')} | dataType: {fld.get('dataType', '')}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Prepare for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\n{rs_id} columns:", df.columns.tolist())
    print(df.head())

# Pick the main record set with most fields/records for further EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None  # fallback for demonstration
main_df = dataframes.get(main_record_set_id, pd.DataFrame())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Select a numeric field for analysis
# Let's find a numeric field from the fields overview
numeric_fields = [fld['@id'] for fld in dataset.fields(record_set=main_record_set_id) if (fld.get('dataType') in ['schema:Integer', 'schema:Float', 'Integer', 'Float'])]
print("Numeric field candidates (@id):", numeric_fields)

# For demonstration, pick the first numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    numeric_field = numeric_field_id
    if numeric_field in main_df.columns:
        # Filter records based on threshold
        threshold = 10
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Grouping by a categorical field
        group_fields = [fld['@id'] for fld in dataset.fields(record_set=main_record_set_id) 
                       if fld.get('dataType') in ['schema:Text', 'Text', 'schema:Boolean', 'Boolean'] and fld['@id'] != numeric_field]
        print("Categorical/group fields candidates (@id):", group_fields)
        if group_fields:
            group_field = group_fields[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())
else:
    print("No numeric fields found in main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization example: histogram of selected numeric field
if main_df.shape[0] > 0 and numeric_fields:
    numeric_field = numeric_fields[0]
    if numeric_field in main_df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(main_df[numeric_field].dropna(), bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

        # Bar plot for group field if available
        group_fields = [fld['@id'] for fld in dataset.fields(record_set=main_record_set_id) 
                       if fld.get('dataType') in ['schema:Text', 'Text', 'schema:Boolean', 'Boolean'] and fld['@id'] != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            if group_field in main_df.columns:
                plt.figure(figsize=(8,5))
                sns.barplot(x=main_df[group_field], y=main_df[numeric_field])
                plt.title(f"Average of {numeric_field} by {group_field}")
                plt.xlabel(group_field)
                plt.ylabel(numeric_field)
                plt.xticks(rotation=45)
                plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, filter, summarize, and visualize the FAIR^2 dataset using `mlcroissant`, referencing data elements by their `@id` fields throughout. Further analyses can be performed based on clinicopathological and molecular features, as well as key categorical variables associated with second primary colorectal cancer among survivors.